# A1.6 · Privilege compromise

**Function A — Securing AI Architectures → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.5 · Tool misuse](https://spbreed.github.io/cyber-commons/lessons/A1.5.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak, SPIFFE/SPIRE |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

An engineer's agent inherits the engineer's standing permissions, because that is the fastest way to make it useful. It now holds production write access at three in the morning, when its principal is asleep and cannot be surprised by anything it does.

## 2 · The framework

```
   human principal                      agent
   +-------------------+                +------------------+
   | repo:write        |  inherits ALL  | repo:write       |
   | deploy:prod       | -------------> | deploy:prod      |
   | secrets:read      |   standing     | secrets:read     |
   +-------------------+                +------------------+
     awake 8 hours/day                    awake 24 hours/day
     asked before acting                  acts on retrieved text
```

**OWASP T3 — Privilege Compromise. LLM06 — Excessive Agency.**

Tool misuse is about what a tool can do. Privilege compromise is about **whose
authority it does it with** — the **identity** component rather than the tools
component.

Three patterns produce it, and all three are things teams do for good reasons.

**Inherited human credentials.** The agent runs with the token of the user who
started it. Convenient, and it means the agent holds every permission that user
holds — including the ones irrelevant to the task, and including the ones they
hold because of a role they were given three years ago.

**Shared service accounts.** Every agent authenticates as `agent-svc`. That
account needs the union of everything any agent ever needs, so each agent holds
the maximum of the set.

**Standing scope.** The grant is permanent because renewing it was operationally
awkward. The authority is therefore present at the moment any injection lands.

What makes this distinct from ordinary over-permissioning is the **direction of
the audit trail**. When a human has too much access and misuses it, the log
names them. When an agent does it, the log names the service account — and the
human who caused it is not in the record at all. The compromise is of privilege
*and* of attribution, at the same time.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

A read-only user asks for something, and the agent has more authority than they do.

In [ ]:
USERS = {"dana":  {"scopes": {"reports:read"}},
         "priya": {"scopes": {"reports:read", "reports:write", "db:admin"}}}

# the agent authenticates as itself, and needs the union of what any user needs
AGENT_SVC = {"name": "agent-svc", "scopes": {"reports:read", "reports:write", "db:admin"}}

AUDIT = []

def call_tool(caller_identity, on_behalf_of, tool, required_scope):
    """Authorization is checked against the CALLER - which is the agent."""
    allowed = required_scope in caller_identity["scopes"]
    AUDIT.append({"actor": caller_identity["name"], "tool": tool,
                  "allowed": allowed})          # note: no human principal
    return allowed

print(f"{'requester':8s}{'their scopes':44s}{'asked for':16s}allowed?")
for user in sorted(USERS):
    ok = call_tool(AGENT_SVC, user, "drop_table", "db:admin")
    print(f"{user:8s}{str(sorted(USERS[user]['scopes'])):44s}{'db:admin':16s}{ok}")

print("\nAUDIT TRAIL")
for a in AUDIT:
    print(f"   actor={a['actor']:10s} tool={a['tool']:12s} allowed={a['allowed']}")

print("\ndana holds reports:read only, and her request reached db:admin.")
print("The authorization decision was made about the agent, not about her.")
print()
print("Now answer 'which user caused the table to be dropped' from that trail.")
print("You cannot: every row says agent-svc. Privilege and attribution failed")
print("in the same step, which is what makes this different from a human with")
print("too much access.")
assert all(a["allowed"] for a in AUDIT)
assert all("dana" not in str(a) for a in AUDIT)

## What you just proved

A user holding only `reports:read` triggers a `db:admin` action, because authorization was evaluated against the shared agent service account rather than the requester — and the audit trail names `agent-svc` on every row, so the human who caused it cannot be recovered from it at all.

## Your turn

Pick one agent and answer two questions: what identity does it authenticate as, and can you name the human behind any single action it took last week. If the second answer is no, you have this risk regardless of how the scopes are set.

---

**Next → [A1.7 · Identity spoofing and impersonation](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*